# OpenFold2/AlphaFold2 Ray Pipeline Example

This notebook demonstrates multi-stages ray pipeline for folding process.

## Imports

### Step 1: Setup Environment

In [ ]:
import os
import time
from pathlib import Path
import ray

from tensorrt_bionemo.pipeline.processor.engine_proc import (
    EngineProcessorConfig, Processor, build_processor)
from tensorrt_bionemo.pipeline.stages.configs import (
    FeatureGeneratorStageConfig, ParserStageConfig, WriterStageConfig)
from tensorrt_bionemo.data.parsers import read_fasta
from tensorrt_bionemo.data.schemas import InputRequest, Polymer, MSARecord
SAMPLE_DIR = Path.cwd().parent / "data" / "samples"

# Set environment variables (uncomment and modify paths as needed)

# PyTorch backend only
ALPHAFOLD2_1_CKPT = "/path/to/alphafold2_1.pt"
# With TRT acceleration
ENGINE_OUTPUT_DIR = "/path/to/alphafold2_1_engines"

MODEL_NAME = "alphafold2_1"
output_dir = "output"

os.environ["ALPHAFOLD2_1_CKPT"] = ALPHAFOLD2_1_CKPT

### Step 2: Build TensorRT Engines (if you want to try with TensorRT engine)

Build optimized TensorRT engines for the Evoformer module. This may take 10-30 minutes depending on your GPU.

In [ ]:
# Build TensorRT engines

# Set build configuration
SAFETENSOR_DIR = "evoformer_safetensors"
MAX_SEQLEN = 1536  # Maximum sequence length to support
MIN_SEQLEN = 16    # Minimum sequence length to support
DTYPE = "float32"

# Step 1: Convert checkpoint to safetensor format
# This converts the PyTorch checkpoint to safetensor format for TRT engine building
!python convert_evoformer_checkpoint.py \
    --dtype ${DTYPE} \
    --model_name ${MODEL_NAME} \
    --output_dir ${SAFETENSOR_DIR} \
    --triangle_attn_backend CUEQUIV

# Step 2: Build TRT engines from safetensor checkpoint
# Build engines for bfloat16 (recommended for better performance)
!trtbnm-build \
    --model ${MODEL_NAME} \
    --module evoformer \
    --checkpoint_dir ${SAFETENSOR_DIR} \
    --max_seqlen ${MAX_SEQLEN} \
    --min_seqlen ${MIN_SEQLEN} \
    --output_dir ${ENGINE_OUTPUT_DIR} \
    --weakly_dtype bfloat16

# After building, set the environment variable to use the engines:
# os.environ["ALPHAFOLD2_1_EVOFORMER_ENGINE"] = ENGINE_OUTPUT_DIR

---

## Running the Pipeline

Once you have your model checkpoint and optionally built TRT engines, you can run the inference pipeline below.

In [32]:
def get_trt_accelerated_configs() -> dict:
    """Check if TRT engines are available from environment variables."""
    if os.path.exists(ENGINE_OUTPUT_DIR):
        return {"evoformer": {"checkpoint": ENGINE_OUTPUT_DIR, "backend": "trt"}}
    return None

In [33]:
def create_sample_requests(repeat: int = 50):
    """Create sample protein folding requests."""
    requests = []
    # sample_ids = ["T1031", "T1033", "T1047s1", "T1096"]
    sample_ids = ["T1047s1"]
    for i in range(repeat):
        for sample_id in sample_ids:
            sequence = read_fasta(str(SAMPLE_DIR / f"{sample_id}.fasta"))["sequences"][0]["sequence"]
            requests.append(
                InputRequest(
                    input_id=f"{sample_id}_{i}",
                    polymers=[Polymer(
                        chain_id="A",
                        sequence=sequence,
                        msas=[MSARecord(path=str(SAMPLE_DIR / "msas" / f"{sample_id}.a3m"))]
                    )]
                )
            )
    
    return requests

In [34]:
def run_pipeline(model_name: str, output_dir: str = "output_structures", trt_acc: dict = None):
    """
    Run the protein structure prediction pipeline.
    
    Args:
        model_name: Model to use (e.g., "alphafold2_1")
        output_dir: Directory to save output PDB files
    """
    
    os.environ["RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION"] = "0.5"
    
    print(f"\n{'='*80}")
    print(f"OpenFold2/AlphaFold2 Structure Prediction Pipeline")
    print(f"{'='*80}")
    print(f"Model: {model_name}")
    
    engine_kwargs = {}
    if trt_acc:
        print(f"Backend: PyTorch + Evoformer TensorRT")
        print(f"Accelerated modules:")
        engine_kwargs["accelerated_configs"] = trt_acc
        for module, path in trt_acc.items():
            print(f"  - {module}: {path}")
    else:
        print(f"Backend: PyTorch (Standard)")
    print(f"{'='*80}\n")
    
    config = EngineProcessorConfig(
        model_source=model_name,
        parser_stage=ParserStageConfig(compute=2),
        feature_generator_stage=FeatureGeneratorStageConfig(compute=4),
        writer_stage=WriterStageConfig(
            compute=2,
            output_path=output_dir,
            format="pdb"
        ),
        engine_kwargs=engine_kwargs
    )
    
    processor = build_processor(config)
    
    requests = create_sample_requests()
    records = [{"record": req, "__record_id": req["input_id"]} for req in requests]
    
    print(f"Processing {len(records)} protein sequences...")
    
    ds = ray.data.from_items(records)
    
    ctx = ray.data.DataContext.get_current()
    ctx.max_errored_blocks = 100
    
    start_time = time.time()
    ds = processor(ds)
    
    success_count = 0
    error_count = 0
    results = []
    
    # materialize the dataset (batching, for the streaming mode don't push the line)
    ds = ds.materialize()
    for row in ds.iter_rows(): # not recomputed if materialized
        if "__inference_error__" in row and row["__inference_error__"]:
            err = row["__inference_error__"]
            if err.get("error_msg"):
                error_count += 1
                print(f"\n❌ Error in {row.get('__record_id', 'unknown')}:")
                print(f"   {err.get('error_msg')}")
        else:
            success_count += 1
            output_path = row.get("output_path")
            if output_path:
                results.append({
                    "id": row.get("__record_id"),
                    "path": output_path
                })
    
    elapsed_time = time.time() - start_time
    
    print(f"\n{'='*80}")
    print(f"Pipeline Results")
    print(f"{'='*80}")
    print(f"Total time: {elapsed_time:.2f} seconds")
    print(f"Time per sequence: {elapsed_time/len(records):.2f} seconds")
    print(f"Successful: {success_count}/{len(records)}")
    print(f"Errors: {error_count}/{len(records)}")
    
    if results:
        print(f"\nOutput structures:")
        for result in results:
            print(f"  ✓ {result['id']}: {result['path']}")
    
    print(f"{'='*80}\n")
    
    return success_count, error_count, elapsed_time

## Configuration and Setup

Set up the model name and output directory. Make sure the required environment variables are set before running the pipeline.

## Run the Pipeline

Execute the pipeline with the configured settings.

In [ ]:
print("Run with torch backend")
try:
    success, errors, elapsed = run_pipeline(MODEL_NAME, output_dir, None)
    
    if errors == 0:
        print(f"✅ Pipeline completed successfully!")
    else:
        print(f"⚠️  Pipeline completed with {errors} errors")
        
except Exception as e:
    print(f"\n❌ Pipeline failed with error:")
    print(f"   {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

2026-01-29 08:21:41,712	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.
2026-01-29 08:21:41,848	INFO logging.py:397 -- Registered dataset logger for dataset dataset_26_0
2026-01-29 08:21:41,851	INFO streaming_executor.py:178 -- Starting execution of Dataset dataset_26_0. Full logs are in /tmp/ray/session_2026-01-29_07-51-59_301434_339793/logs/ray-data
2026-01-29 08:21:41,851	INFO streaming_executor.py:179 -- Execution plan of Dataset dataset_26_0: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(ParserUDF)] -> ActorPoolMapOperator[MapBatches(TokenizerUDF)] -> ActorPoolMapOperator[MapBatches(FeatureGeneratorUDF)] -> ActorPoolMapOperator[MapBatches(FoldingEngineUDF)] -> ActorPoolMapOperator[MapBatches(WriterUDF)]


Run with torch backend

OpenFold2/AlphaFold2 Structure Prediction Pipeline
Model: alphafold2_1
Backend: PyTorch (Standard)

Processing 50 protein sequences...


2026-01-29 08:22:16,217	WARNING resource_manager.py:761 -- Cluster resources are not enough to run any task from ActorPoolMapOperator[MapBatches(ParserUDF)]. The job may hang forever unless the cluster scales up.
2026-01-29 08:22:16,246	INFO progress_bar.py:213 -- === Ray Data Progress {MapBatches(ParserUDF)} ===
2026-01-29 08:22:16,247	INFO progress_bar.py:215 -- MapBatches(ParserUDF): Tasks: 2; Actors: 2 (running=1, restarting=0, pending=1); Queued blocks: 48 (0.0B); Resources: 1.0 CPU, 384.0MiB object store; [all objects local]: Progress Completed 0 / ?
2026-01-29 08:22:16,248	INFO progress_bar.py:213 -- === Ray Data Progress {MapBatches(TokenizerUDF)} ===
2026-01-29 08:22:16,249	INFO progress_bar.py:215 -- MapBatches(TokenizerUDF): Tasks: 0; Actors: 1; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store; [all objects local]: Progress Completed 0 / ?
2026-01-29 08:22:16,249	INFO progress_bar.py:213 -- === Ray Data Progress {MapBatches(FeatureGeneratorUDF)} ===
2026-01-29 


Pipeline Results
Total time: 224.45 seconds
Time per sequence: 4.49 seconds
Successful: 0/50
Errors: 0/50

✅ Pipeline completed successfully!


/home/dev_user/.local/lib/python3.12/site-packages/ray/data/_internal/execution/streaming_executor.py:345: ResourceWarning: unclosed file <_io.TextIOWrapper name='/tmp/ray/session_2026-01-29_07-51-59_301434_339793/logs/ray-data/ray-data-dataset_26_0.log' mode='a' encoding='UTF-8'>
  unregister_dataset_logger(self._dataset_id)


In [36]:
print("Run with torch + Evoformer TRT backend")
trt_acc = get_trt_accelerated_configs()
try:
    success, errors, elapsed = run_pipeline(MODEL_NAME, output_dir, trt_acc)
    
    if errors == 0:
        print(f"✅ Pipeline completed successfully!")
    else:
        print(f"⚠️  Pipeline completed with {errors} errors")
        
except Exception as e:
    print(f"\n❌ Pipeline failed with error:")
    print(f"   {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

2026-01-29 08:34:51,006	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.
2026-01-29 08:34:51,140	INFO logging.py:397 -- Registered dataset logger for dataset dataset_41_0
2026-01-29 08:34:51,141	INFO logging.py:405 -- dataset_41_0 registers for logging while another dataset dataset_34_0 is also logging. For performance reasons, we will not log to the dataset dataset_41_0 until it is the only active dataset.
2026-01-29 08:34:51,143	INFO streaming_executor.py:178 -- Starting execution of Dataset dataset_41_0. Full logs are in /tmp/ray/session_2026-01-29_07-51-59_301434_339793/logs/ray-data
2026-01-29 08:34:51,143	INFO streaming_executor.py:179 -- Execution plan of Dataset dataset_41_0: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(ParserUDF)] -> ActorPoolMapOperator[MapBatches(TokenizerUDF)] -> ActorPoolMapOperator[MapBatches(FeatureGeneratorUDF)] -> ActorPoolMapOperator[MapBatches(FoldingEngineUDF)] -> ActorPoolMapOperator[MapBatches(WriterUD

Run with torch + Evoformer TRT backend

OpenFold2/AlphaFold2 Structure Prediction Pipeline
Model: alphafold2_1
Backend: PyTorch + Evoformer TensorRT
Accelerated modules:
  - evoformer: {'checkpoint': '/workspace/tensorrt_bionemo/examples/openfold2/alphafold2_1_engines', 'backend': 'trt'}

Processing 50 protein sequences...


2026-01-29 08:35:31,486	WARNING resource_manager.py:761 -- Cluster resources are not enough to run any task from ActorPoolMapOperator[MapBatches(ParserUDF)]. The job may hang forever unless the cluster scales up.
2026-01-29 08:35:31,518	INFO progress_bar.py:213 -- === Ray Data Progress {MapBatches(ParserUDF)} ===
2026-01-29 08:35:31,519	INFO progress_bar.py:215 -- MapBatches(ParserUDF): Tasks: 2; Actors: 2 (running=1, restarting=0, pending=1); Queued blocks: 48 (0.0B); Resources: 1.0 CPU, 384.0MiB object store; [all objects local]: Progress Completed 0 / ?
2026-01-29 08:35:31,520	INFO progress_bar.py:213 -- === Ray Data Progress {MapBatches(TokenizerUDF)} ===
2026-01-29 08:35:31,520	INFO progress_bar.py:215 -- MapBatches(TokenizerUDF): Tasks: 0; Actors: 1; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store; [all objects local]: Progress Completed 0 / ?
2026-01-29 08:35:31,521	INFO progress_bar.py:213 -- === Ray Data Progress {MapBatches(FeatureGeneratorUDF)} ===
2026-01-29 


Pipeline Results
Total time: 187.00 seconds
Time per sequence: 3.74 seconds
Successful: 0/50
Errors: 0/50

✅ Pipeline completed successfully!
